# 02 | Study area, acquisition geometry and reference grid
We train on a regional window around Bagré, rather than the tiny farm pilot alone. SMAP's enhanced 9 km grid is still derived from a much coarser radiometer footprint; it is not field truth. Earth Engine ingests the product in geographic coordinates. We use its actual ingested affine grid and preserve its identity. We use 2024 for training/validation and 2025 for temporal testing.

The wider region includes a margin; the inference/evaluation core avoids truncated coarse-cell footprints. The pilot is shown only for local interpretation.

## Setup (CPU, standard RAM)
Run this cell first in each notebook. The project uses small monthly raster tiles and tabular ML. If Colab requests a restart, use Runtime → Restart session and rerun from the beginning. Do not change NumPy manually. Files persist in Drive.

In [ ]:
%pip -q install earthengine-api rasterio pyproj scikit-learn joblib
import sys,json
from pathlib import Path
from importlib.metadata import version
for pkg,mod in [('numpy','numpy'),('scikit-learn','sklearn'),('rasterio','rasterio')]:
    loaded=sys.modules.get(mod)
    if loaded is not None and getattr(loaded,'__version__',version(pkg))!=version(pkg):
        raise RuntimeError(f'{pkg} changed. Restart session and rerun.')
from google.colab import drive
drive.mount('/content/drive')
ROOT=Path('/content/drive/MyDrive/Bagre_SAR_Soil_Moisture')
assert (ROOT/'code'/'sar_workflow.py').is_file(), 'Check Drive folder and shared code.'
sys.path.insert(0,str(ROOT/'code'))
import sar_workflow as sw
import numpy as np, pandas as pd, matplotlib.pyplot as plt, joblib
for sub in ['data','models','outputs','figures','qgis']: (ROOT/sub).mkdir(exist_ok=True)
print('NumPy:',version('numpy'),'scikit-learn:',version('scikit-learn'))

## Connect to Earth Engine
Your existing Cloud project is prefilled. Sign in with the registered Google account when asked. Earth Engine runs the satellite processing remotely.

In [ ]:
import ee_workflow as ew
import ee
PROJECT='YOUR_EARTH_ENGINE_PROJECT_ID'
ew.initialize(PROJECT)

## Orbit audit (before looking at soil moisture)
Restrict to IW, 10 m VV+VH. Do not mix relative orbits or ascending/descending geometries; select the track with the most months of pilot coverage, then regional footprint coverage and scene count. These are scene footprints, not valid-data coverage.

In [ ]:
tracks=ew.track_audit(); display(tracks)
tracks.to_csv(ROOT/'outputs'/'orbit_audit.csv',index=False)
chosen=tracks.iloc[0]
fine=sw.grid(sw.REGION,sw.CRS,250)
smap_proj=ee.Image(ew.moisture_collection().first()).select('soil_moisture_am').projection().getInfo()
coarse=sw.native_grid(smap_proj)
print('Fine grid:',fine,'SMAP ingested projection:',smap_proj,'SMAP subset:',coarse)
assert fine['width']*fine['height']*4*4<30_000_000,'Fine download exceeds intended budget'
conf=dict(pass_name=chosen['pass'],orbit=int(chosen.orbit),fine=fine,coarse=coarse,smap_projection=smap_proj)
conf['pass']=conf.pop('pass_name')
path=ROOT/'data'/'configuration.json'
if path.exists():
    assert json.loads(path.read_text())==conf, 'Configuration changed; use a new folder or move the prior cached inputs before rerunning.'
path.write_text(json.dumps(conf,indent=2))

## Monthly inventory
Counts are scene/granule counts. More scenes do not guarantee good pixels; we inspect actual masks next. Missing months are preserved rather than imputed.

In [ ]:
track=ew.track_collection(conf['pass'],conf['orbit'])
def hist(col):
    return col.map(lambda im:im.set('month',im.date().format('YYYY-MM'))).aggregate_histogram('month').getInfo()
s1h,s2h,smh=hist(track),hist(ew.optical_collection()),hist(ew.moisture_collection())
audit=pd.DataFrame({'month':sw.MONTHS.strftime('%Y-%m')})
for name,h in [('sar_scenes',s1h),('optical_granules',s2h),('smap_images',smh)]: audit[name]=audit.month.map(h).fillna(0).astype(int)
display(audit); audit.to_csv(ROOT/'outputs'/'monthly_inventory.csv',index=False)

## Static land cover and first-month smoke test
WorldCover 2021 is only a static land-cover proxy. We calculate land fraction (all classes except permanent water) and crop fraction at 250 m. The dry-season first month demonstrates the end-to-end request. Monthly SAR averaging is in linear power. SCL 4/5 (vegetation/bare ground) limits optical contamination, but cloud/shadow errors remain.

In [ ]:
sw.download(ew.land_image(fine),ROOT/'data'/'land.tif',fine,['land_fraction','crop_fraction'])
ew.acquire_month(ROOT,sw.MONTHS[0],conf)
land=sw.read_tif(ROOT/'data'/'land.tif',fine,['land_fraction','crop_fraction'])
sar=sw.read_tif(ROOT/'data'/'sar_2024-01.tif',fine,sw.S1)
optical=sw.read_tif(ROOT/'data'/'optical_2024-01.tif',fine,sw.S2)
smap=sw.read_tif(ROOT/'data'/'smap_2024-01.tif',coarse,['sm','smap_days'])
table,valid=sw.summarize_month(sar,optical,land,smap,fine,coarse,'2024-01')
print('Eligible coarse cells:',table.eligible.sum(),'of',table.core.sum(),'core cells')
fig,axs=plt.subplots(1,4,figsize=(15,4),constrained_layout=True)
for ax,a,title in zip(axs,[sw.db(sar[0]),optical[0],smap[0],land[1]],['VV (dB of mean power)','NDVI','SMAP m³/m³','Crop fraction']):
    h=ax.imshow(a); ax.set_title(title); ax.axis('off'); fig.colorbar(h,ax=ax,shrink=.65)
fig.savefig(ROOT/'figures'/'02_smoke_test.png',dpi=150); plt.show()
assert table.core.sum()>0 and np.isfinite(sar[0]).any() and np.isfinite(smap[0]).any(), 'No first-month SAR or SMAP values; report output before continuing.'

**Checkpoint:** check the track, dimensions and January map. Report missingness and any HTTP error before proceeding. The regional grid is UTM 30N at 250 m; the SMAP grid is the ingested geographic grid, not 250 m.